# 📘 Introducción para gente que acaba de empezar

## ¿Qué vamos a aprender aquí?

Este notebook continúa el anterior (notebook 11, sobre Word2Vec). Vamos a aprender un modelo llamado **FastText** que **resuelve un problema que vimos antes**: las palabras desconocidas.

### Recuerda el problema del notebook 11

En Word2Vec, cuando pedimos `most_similar('asereje')`, nos dio un **error feo**:

```
KeyError: "Key 'asereje' not present in vocabulary"
```

¿Por qué? Porque Word2Vec aprende un vector **por palabra entera**. Si nunca vio "asereje" durante el entrenamiento, no tiene vector para ella → falla.

### FastText resuelve esto

FastText es **una mejora de Word2Vec**, también inventada por Tomas Mikolov (mismo autor) y su equipo en Facebook (2016). La idea es **brillante en su simplicidad**:

> 🎯 **En lugar de aprender embeddings de palabras enteras, FastText aprende embeddings de TROZOS de palabras (n-gramas de caracteres).**

Cuando le pidamos una palabra nueva, **construye su vector** sumando los vectores de sus trozos.

### Analogía sencilla

Imagina que tienes que aprender los nombres de TODAS las personas del mundo:

- **Word2Vec** = memorizar cada nombre individualmente. Si llega alguien nuevo, no lo conoces.
- **FastText** = aprender **sílabas** (`ma`, `ri`, `lu`, `pe`...). Cuando llega alguien nuevo (María, Pedro, Lucía...), puedes leer su nombre combinando sílabas que ya conoces.

Es una idea simple pero **revolucionaria** para idiomas con mucha morfología (español, alemán, finés...) y para corpus con muchos typos.

### Lo que veremos en este notebook

1. Cargaremos el **mismo corpus de Los Simpsons** ya limpio (del notebook anterior).
2. Configuraremos FastText con sus hiperparámetros (incluyendo `min_n` y `max_n`, **nuevos** respecto a Word2Vec).
3. Entrenaremos el modelo.
4. Probaremos los mismos ejemplos que con Word2Vec y **compararemos resultados**.
5. **Demostraremos que ahora `most_similar('asereje')` SÍ funciona** sin error.

### Si no entiendes algo, no pasa nada

Este notebook usa código que puede parecer complejo, pero **la idea es simple**. Vamos a explicar cada línea.


# FastText

A diferencia de Word2Vec, que trabaja a nivel de palabra, FastText trata de capturar la información morfológica de las palabras.

>*"[...] we propose a new approach **based on the skipgram model, where each word is represented as a bag of character n-grams**. A vector representation is associated to each character n-gram; words being represented as the sum of these representations. [...]"* <br>(Mikolov et al., Enriching Word Vectors with Subword Information, https://arxiv.org/pdf/1607.04606.pdf)

De esta manera, una palabra quedará representada por sus n-grams.

El tamaño de los n-grams deberá ser definido como hiperparámetro
- min_n: valor mínimo de _n_ a considerar
- max_n: valor máximo de _n_ a considerar

Ejemplo:
>*"Me gusta el procesado del lenguaje natural"*
>* Ejemplo de *skip-gram* pre-procesado con una ventana de contexto de 2 palabras
>
>$w_{target} =$ "procesado" &emsp;$w_{context} =$ ["gusta", "el", "del", "lenguaje"]
>
>     ("procesado", "gusta")
>
> Descomoposición de n-grams con min_n=3 and max_n=4:
>
>"procesado" = ["$<$pr", "pro", ..., "ado", "do$>$", "$<$pro", "roce", ..., "sado", "ado$>$"]
>
>* De este modo, la similitud será: <br><br>
>&emsp;$\boxed{s(w_{target}, w_{context}) = \sum_{g \in G_{w_{target}}}z_{g}^T v_{w_{context}}}$, where $G_{w_{target}}\subset\{g_{1}, ..., g_{G}\}$

### Explicación intuitiva: ¿qué son los n-gramas de caracteres?

La celda de arriba tiene mucha matemática y notación. Vamos a desmenuzarla.

#### Qué es un n-grama de caracteres

Un **n-grama de caracteres** es un trozo consecutivo de letras de longitud N.

Para la palabra `homer`:

| Tipo | n-gramas |
| --- | --- |
| 3-gramas | `<ho`, `hom`, `ome`, `mer`, `er>` |
| 4-gramas | `<hom`, `home`, `omer`, `mer>` |
| 5-gramas | `<home`, `homer`, `omer>` |

Los símbolos `<` y `>` marcan **inicio** y **fin** de palabra (para distinguir, por ejemplo, `<her` al principio de `her>` al final).

#### Por qué esto resuelve OOV

Para la palabra **nueva** `homers`:
- 3-gramas: `<ho`, `hom`, `ome`, `mer`, `ers`, `rs>`
- 4-gramas: `<hom`, `home`, `omer`, `mers`, `ers>`

¡Casi todos están en común con `homer`! FastText suma los vectores de esos n-gramas y obtiene un vector para `homers` muy parecido al de `homer`. **Sin necesidad de haberla visto durante el entrenamiento**.

#### Hiperparámetros nuevos: `min_n` y `max_n`

| Parámetro | Significado | Valor típico |
| --- | --- | --- |
| `min_n` | Longitud mínima del n-grama | 3 |
| `max_n` | Longitud máxima del n-grama | 6 |

Con `min_n=3, max_n=6`, FastText considera 3-gramas, 4-gramas, 5-gramas Y 6-gramas. Más rango = más capacidad de capturar morfología, pero también más memoria.

#### La fórmula matemática (puedes saltártela)

```
s(target, context) = Σ z_g^T · v_context
                     g ∈ G_target
```

En lenguaje humano: *"la similitud entre una palabra target y una palabra context se calcula sumando los productos de los vectores de los n-gramas de target con el vector de context"*.

Es solo "suma de productos" — no te preocupes si no lo entiendes a la primera. **Lo importante es saber que cada palabra es la suma de sus trozos**.

#### Por qué FastText es la evolución natural

| Característica | Word2Vec | FastText |
| --- | --- | --- |
| Trabaja con | Palabras enteras | n-gramas de caracteres |
| OOV (palabras nuevas) | ❌ Error | ✅ Construye el vector |
| Typos (`homerr`) | Sin vector | Vector similar a `homer` |
| Variantes morfológicas | Independientes | Comparten información |
| Idiomas con flexión (español, alemán) | Mediocre | Excelente |
| Memoria | Menor | Mayor (almacena n-gramas) |
| Velocidad de entrenamiento | Más rápida | Más lenta |

> 💡 **Regla práctica**: si tu corpus tiene typos, slang, o es de un idioma con mucha flexión gramatical, **usa FastText**. Para inglés simple y limpio, Word2Vec puede bastar.


## Palabras más similares

### Sobre este encabezado

El encabezado *"Palabras más similares"* está colocado pronto, pero las consultas de palabras similares (`most_similar`) vendrán **más adelante**, después de entrenar el modelo.

Es un anticipo de lo que veremos: igual que en el notebook 11, exploraremos qué palabras se parecen a `homer`, `marge`, `bart`...


In [1]:
!pip install gensim spacy numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 52.8 MB/s eta 0:00:00


### Las dependencias

```python
!pip install gensim spacy numpy
```

Estamos instalando **tres paquetes** en una sola línea (separados por espacios):

| Paquete | Para qué sirve |
| --- | --- |
| **gensim** | Biblioteca con la implementación de FastText. La usaremos para entrenar el modelo. |
| **spacy** | Procesamiento de lenguaje natural avanzado (lematización, tokenización). |
| **numpy** | Cálculo numérico. Casi todas las librerías de ML lo usan internamente. |

> 💡 **Nota**: `gensim`, `spacy` y `numpy` probablemente **ya estén instalados** en Colab. Este `pip install` solo se asegura de que estén disponibles. Si ya están, simplemente confirma.

### El símbolo `!` al principio

Recordatorio: el `!` le dice al notebook *"esto NO es Python, ejecútalo como un comando del sistema"*. Así podemos llamar a `pip` sin salir del notebook.


In [2]:
def print_sim_words(word, model1, model2):
    query = "Most similar to {}".format(word)
    print(query)
    print("-"*len(query))
    for (sim1, sim2) in zip(model1.wv.most_similar(word), model2.wv.most_similar(word)):
        print("{}:{}{:.3f}{}{}:{}{:.3f}".format(sim1[0],
                                               " "*(20-len(sim1[0])),
                                               sim1[1],
                                               " "*10,
                                               sim2[0],
                                               " "*(20-len(sim2[0])),
                                               sim2[1]))
    print("\n")

### Una función auxiliar para comparar dos modelos

```python
def print_sim_words(word, model1, model2):
    ...
```

#### Qué hace la función

Imprime, **lado a lado**, las palabras más similares según **dos modelos diferentes**. Útil para comparar (por ejemplo) Word2Vec vs FastText.

#### Línea por línea

```python
def print_sim_words(word, model1, model2):
    query = "Most similar to {}".format(word)
    print(query)
    print("-"*len(query))
```

- Define la función con 3 parámetros: una palabra y dos modelos.
- Imprime un título tipo `"Most similar to homer"`.
- Imprime una línea de guiones debajo (decorativo, del mismo largo que el título).

```python
    for (sim1, sim2) in zip(model1.wv.most_similar(word), model2.wv.most_similar(word)):
```

- `model1.wv.most_similar(word)` → lista de las 10 palabras más similares según el modelo 1.
- `model2.wv.most_similar(word)` → idem según el modelo 2.
- `zip(...)` → empareja la primera de cada lista, la segunda de cada lista, etc.
- `sim1` y `sim2` son tuplas `(palabra, similitud)`.

```python
        print("{}:{}{:.3f}{}{}:{}{:.3f}".format(sim1[0], " "*(20-len(sim1[0])), sim1[1], " "*10, sim2[0], " "*(20-len(sim2[0])), sim2[1]))
```

Esto da miedo, pero solo formatea la salida en columnas alineadas:
- `sim1[0]` → la palabra del modelo 1.
- `" "*(20-len(sim1[0]))` → rellena con espacios hasta longitud 20 (alineación).
- `sim1[1]` con `:.3f` → similitud con 3 decimales.
- `" "*10` → 10 espacios entre los dos modelos.
- Repite para el modelo 2.

#### Detalle didáctico

⚠️ **La función está definida pero NO se llega a usar en el notebook**. Está ahí para que tú la uses si entrenas un modelo Word2Vec en paralelo y quieres comparar. Es un "regalo" del notebook que el código no aprovecha.

#### Cómo la usarías

Si en un notebook anterior guardaste un Word2Vec llamado `w2v`:

```python
print_sim_words('homer', w2v, ft_sg)
```

Te imprimiría una tabla bonita comparando ambos modelos.


## Importamos las librerías

### Las importaciones que necesitamos


In [3]:

from gensim.models import FastText
from gensim.models.word2vec import LineSentence
from gensim.models.phrases import Phrases, Phraser

### Qué importamos y para qué

```python
from gensim.models import FastText
from gensim.models.word2vec import LineSentence
from gensim.models.phrases import Phrases, Phraser
```

| Import | Para qué sirve |
| --- | --- |
| **`FastText`** | La clase del modelo que vamos a entrenar. ¡La estrella del notebook! |
| `LineSentence` | Helper para leer corpus desde archivo (línea = frase). **No se usa aquí**, pero se importa por costumbre. |
| `Phrases`, `Phraser` | Para detectar bigramas frecuentes (ej: `new_york` como un solo token). **Tampoco se usan**. |

> 💡 **Buena práctica**: cuando un import no se usa, en un proyecto real lo eliminarías. Aquí están "por si acaso" del que escribió el notebook.


## Lectura de datos

### Reutilizamos el corpus del notebook anterior

En el notebook 11 limpiamos el dataset de Los Simpsons (regex, lematización, eliminación de stopwords) y lo guardamos como `df_clean_simpsons.csv`.

Ahora cargamos directamente ese CSV limpio para no repetir el preprocesado (que tardaba 3-4 minutos).

> 🎯 **Principio profesional**: separa **preprocesado** y **entrenamiento** en notebooks/scripts distintos. El preprocesado es lento pero se hace pocas veces; el entrenamiento se hace muchas veces probando hiperparámetros.


In [4]:
!pip install unzip
!unzip df_clean_simpsons.csv.zip

  Preparing metadata (setup.py) ... done
  Created wheel for unzip: filename=unzip-1.0.0-py3-none-any.whl size=1281 sha256=89437fb810ddf7055b0b42b140a1617ee99f76813c24096b0d57c6c3133d2874
  Stored in directory: /root/.cache/pip/wheels/fb/5b/81/0f3e1e533b52883f88ab978178c15627a4fce4c13f74911dce
Successfully built unzip
Archive:  df_clean_simpsons.csv.zip
replace df_clean_simpsons.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: df_clean_simpsons.csv   
  inflating: __MACOSX/._df_clean_simpsons.csv  


### Descomprimir el archivo del corpus

```python
!pip install unzip
!unzip df_clean_simpsons.csv.zip
```

- **Primera línea**: `!pip install unzip` no instala nada útil (no existe un paquete pip "unzip"). Es un error inofensivo del notebook original.
- **Segunda línea**: `!unzip` SÍ funciona — es un comando del sistema Linux/Colab que descomprime el `.zip` y deja `df_clean_simpsons.csv` listo para leer.


In [5]:
import pandas as pd
df_clean = pd.read_csv('./df_clean_simpsons.csv')

### Cargar el corpus limpio

```python
import pandas as pd
df_clean = pd.read_csv('./df_clean_simpsons.csv')
```

- **`pd.read_csv(...)`** abre el CSV y lo convierte en un **DataFrame** (tabla de pandas).
- El archivo tiene una columna llamada `clean` con las frases ya preprocesadas.

#### Lo que contiene el DataFrame

Si lo inspeccionamos, veríamos algo como:

| | clean |
| --- | --- |
| 0 | homer love duff beer |
| 1 | marge worry kid school |
| 2 | bart skateboard street |
| ... | ... |

Cada fila es una frase ya limpia: solo palabras útiles, sin puntuación, lematizada.


In [6]:

sent = [row.split() for row in df_clean['clean']]

### Preparar el formato para FastText

```python
sent = [row.split() for row in df_clean['clean']]
```

#### Qué hace esta línea

Recorre cada fila del DataFrame y la divide en una **lista de palabras**.

Si `df_clean['clean']` contiene:
```
['homer love duff', 'marge worry kid', ...]
```

Entonces `sent` será:
```python
[['homer', 'love', 'duff'], ['marge', 'worry', 'kid'], ...]
```

#### ¿Por qué este formato?

FastText (igual que Word2Vec) **espera una lista de listas de palabras** como input. Cada lista interna representa una frase. Es el formato estándar de gensim.

#### Comprensión de listas

```python
[row.split() for row in df_clean['clean']]
```

Es Python avanzado pero útil. Equivale a:

```python
sent = []
for row in df_clean['clean']:
    sent.append(row.split())
```

Una sola línea hace lo mismo que tres líneas con un bucle.


## Hyperparameters

### Hiperparámetros: las palancas que controlamos

Igual que con Word2Vec, FastText tiene una serie de **hiperparámetros** (decisiones que tomamos antes de entrenar). Los más relevantes son:

| Hiperparámetro | Word2Vec | FastText | Nuevo |
| --- | --- | --- | --- |
| `sg` | ✅ | ✅ | - |
| `vector_size` | ✅ | ✅ | - |
| `min_count` | ✅ | ✅ | - |
| `window` | ✅ | ✅ | - |
| `negative` | ✅ | ✅ | - |
| **`min_n`** | ❌ | ✅ | 🆕 Longitud mínima de n-grama |
| **`max_n`** | ❌ | ✅ | 🆕 Longitud máxima de n-grama |

Los dos últimos son **lo nuevo** de FastText. Son los que le dan su superpoder de manejar OOV.


In [7]:
sg_params = {
    'sg': 1,
    'vector_size': 300,
    'min_count': 5,
    'window': 5,
    'hs': 0,
    'negative': 20,
    'workers': 4,
    'min_n': 3,
    'max_n': 6
}



### El diccionario de hiperparámetros

```python
sg_params = {
    'sg': 1,
    'vector_size': 300,
    'min_count': 5,
    'window': 5,
    'hs': 0,
    'negative': 20,
    'workers': 4,
    'min_n': 3,
    'max_n': 6
}
```

#### ¿Por qué un diccionario y no parámetros sueltos?

Usar un **dict** y luego pasarlo con `**sg_params` permite:
- **Documentar** todos los hiperparámetros en un solo bloque.
- **Reutilizar** la configuración para crear otros modelos.
- **Compararlos** fácilmente si pruebas variantes.

#### Cada parámetro explicado

| Parámetro | Valor | Significado |
| --- | --- | --- |
| `sg=1` | Skip-Gram | Algoritmo: 1 = Skip-Gram, 0 = CBOW. **Skip-Gram funciona mejor con palabras raras** y morfología. |
| `vector_size=300` | 300 | Dimensiones del embedding. Estándar de industria. |
| `min_count=5` | 5 | Solo palabras con ≥5 apariciones. **Más permisivo que en el notebook 11** (que era 20). |
| `window=5` | 5 | Mira 5 palabras a cada lado. **Más amplia** que el notebook 11 (que era 2). |
| `hs=0` | Sin softmax jerárquica | Indica que usaremos negative sampling. |
| `negative=20` | 20 | Negative sampling: 20 palabras "negativas" por positiva. |
| `workers=4` | 4 | Núcleos de CPU para entrenar en paralelo. |
| **`min_n=3`** | 3 | **Longitud mínima del n-grama**. |
| **`max_n=6`** | 6 | **Longitud máxima del n-grama**. |

#### Comparación con la celda 11 vs la 14

⚠️ **Detalle importante**: este mismo `sg_params` se define **dos veces** en el notebook (celdas 11 y 14). La celda 14 lo redefine y luego crea el modelo. La celda 11 es **redundante** (sobra). En código limpio se eliminaría.

#### Por qué `min_n=3, max_n=6`

Es el rango clásico:
- **3** captura sufijos comunes (`-ing`, `-ed`, `-ly`).
- **6** captura raíces largas (`fight`, `learn`, `start`).
- Más allá de 6, los n-gramas se hacen casi palabras enteras (poco útil).

> 💡 Para idiomas con palabras muy largas (alemán, finés), podrías subir `max_n` a 8 o 10.


## Inicializamos el objeto FastText

### Crear el objeto FastText

Como hicimos con Word2Vec, primero **creamos** el modelo (sin entrenarlo todavía) y le pasamos los hiperparámetros.

A diferencia de Word2Vec, la documentación de FastText tiene **muchos más parámetros** porque maneja también n-gramas. Por eso la celda siguiente llama a `help(FastText)` para ver toda la documentación.


In [8]:
help(FastText)

Help on class FastText in module gensim.models.fasttext:

class FastText(gensim.models.word2vec.Word2Vec)
 |  FastText(sentences=None, corpus_file=None, sg=0, hs=0, vector_size=100, alpha=0.025, window=5, min_count=5, max_vocab_size=None, word_ngrams=1, sample=0.001, seed=1, workers=3, min_alpha=0.0001, negative=5, ns_exponent=0.75, cbow_mean=1, hashfxn=<built-in function hash>, epochs=5, null_word=0, min_n=3, max_n=6, sorted_vocab=1, bucket=2000000, trim_rule=None, batch_words=10000, callbacks=(), max_final_vocab=None, shrink_windows=True)
 |
 |  Method resolution order:
 |      FastText
 |      gensim.models.word2vec.Word2Vec
 |      gensim.utils.SaveLoad
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(self, sentences=None, corpus_file=None, sg=0, hs=0, vector_size=100, alpha=0.025, window=5, min_count=5, max_vocab_size=None, word_ngrams=1, sample=0.001, seed=1, workers=3, min_alpha=0.0001, negative=5, ns_exponent=0.75, cbow_mean=1, hashfxn=<built-in function has

### ¿Qué es `help()`?

```python
help(FastText)
```

`help()` es una **función incorporada** de Python que imprime la **documentación** de cualquier cosa: una función, una clase, un módulo.

#### Cuándo usarla

- **Cuando no recuerdas qué parámetros acepta** una función.
- **Cuando quieres ver ejemplos** (si la documentación los tiene).
- **Cuando estás aprendiendo** una librería nueva.

#### Alternativas

| Forma | Cuándo |
| --- | --- |
| `help(FastText)` | Imprime todo en el notebook. |
| `FastText?` | Lo mismo, pero en Jupyter (más compacto). |
| `FastText??` | Muestra incluso el código fuente. |
| Ir a la documentación web | Para leer con calma. |

#### Lo que veremos

Aparecerá una pared de texto con todos los parámetros de `FastText`, sus tipos esperados y descripciones. Es **largo y técnico**, pero útil cuando necesitas un parámetro concreto.


In [9]:
sg_params = {
    'sg': 1,
    'vector_size': 300,
    'min_count': 5,
    'window': 5,
    'hs': 0,
    'negative': 20,
    'workers': 4,
    'min_n': 3,
    'max_n': 6
}

# Skip Gram
ft_sg = FastText(**sg_params)

### Crear el modelo de FastText con Skip-Gram

```python
ft_sg = FastText(**sg_params)
```

#### El operador `**` (desempaquetar diccionario)

`**sg_params` significa: *"desempaqueta el diccionario y pasa cada clave como un parámetro nombrado"*.

Es decir, **estas dos líneas son equivalentes**:

```python
# Forma 1 (verbose)
ft_sg = FastText(sg=1, vector_size=300, min_count=5, window=5, ...)

# Forma 2 (compacta, con **)
ft_sg = FastText(**sg_params)
```

#### Por qué se llama `ft_sg`

- **`ft`** = FastText.
- **`sg`** = Skip-Gram (el algoritmo que estamos usando, `sg=1`).

Si entrenáramos otro modelo con CBOW, lo llamaríamos `ft_cbow`. Es una convención clara.

#### Qué hace esta línea

**Crea el objeto del modelo** y reserva memoria para los embeddings, pero **NO entrena nada todavía**. Solo es la "ficha técnica".

Aún hay que:
1. **Construir el vocabulario** (`build_vocab`).
2. **Entrenar** (`train`).


## Construímos el vocabulario

### Construir el vocabulario antes de entrenar

Igual que con Word2Vec, FastText sigue 3 pasos:

1. **Crear el modelo** ✅ (ya hecho)
2. **Construir el vocabulario** ← estamos aquí
3. **Entrenar**

#### Por qué construir el vocabulario primero

FastText necesita:
- **Saber qué palabras existen** y cuántas veces aparece cada una.
- **Generar todos los n-gramas** para esas palabras.
- **Reservar memoria** para los embeddings de palabras y n-gramas.

Es el **censo previo** del corpus. Sin él, no sabría qué entrenar.


In [10]:
# Skip Gram
ft_sg.build_vocab(sent)



### Construyendo el vocabulario

```python
ft_sg.build_vocab(sent)
```

Esta línea hace **mucho trabajo por detrás**:

1. Recorre todas las frases de `sent`.
2. Cuenta cuántas veces aparece cada palabra.
3. Filtra las que aparecen menos de `min_count=5` veces.
4. **Para cada palabra que queda, genera todos sus n-gramas** entre `min_n=3` y `max_n=6`.
5. Crea un mapeo `n-grama → índice` (para acceso rápido).
6. Reserva matrices vacías de embeddings.

#### Es mucho más complejo que en Word2Vec

En Word2Vec, solo había que contar palabras. En FastText, **además** hay que generar y almacenar todos los n-gramas. Por eso usa más memoria.

> Si entrenaras con un corpus enorme (millones de frases), este paso podría tardar bastante. Aquí con ~86k frases es rápido.


In [11]:
print('Vocabulario compuesto por {} palabras'.format(len(ft_sg.wv.key_to_index)))


Vocabulario compuesto por 8770 palabras


### Tamaño del vocabulario: 8770 palabras

#### Comparación con Word2Vec del notebook anterior

| Modelo | `min_count` | Vocabulario |
| --- | --- | --- |
| Word2Vec (notebook 11) | 20 | 3.239 |
| **FastText (este notebook)** | **5** | **8.770** |

#### Por qué tantas más palabras

Porque bajamos `min_count` de 20 a 5. Cualquier palabra que aparezca **al menos 5 veces** entra al vocabulario.

Resultado: **2.7 veces más palabras**, incluyendo nombres propios menos frecuentes, palabras temáticas raras, etc.

#### Ojo: además del vocabulario explícito, hay n-gramas

FastText también almacena vectores para **n-gramas** internamente. Por defecto guarda **2 millones de buckets** para n-gramas (configurable con `bucket=2000000`). Eso es lo que le permite responder por cualquier palabra que no esté en el vocabulario.


## Entrenamos los pesos de los embeddings

### Por fin: entrenar el modelo

Hasta ahora:
- ✅ Cargamos corpus.
- ✅ Configuramos hiperparámetros.
- ✅ Creamos el modelo.
- ✅ Construimos vocabulario y n-gramas.

Ahora: **entrenamos**. Aquí es donde el modelo realmente **aprende** los vectores.


In [12]:
# Skip Gram


ft_sg.train(sent, total_examples=len(sent), epochs=20)


(9125274, 10741780)

### Entrenamiento completado

```python
ft_sg.train(sent, total_examples=len(sent), epochs=20)
```

#### Parámetros

| Parámetro | Significado |
| --- | --- |
| `sent` | El corpus (lista de listas de palabras). |
| `total_examples=len(sent)` | Número total de frases (para que gensim muestre progreso). |
| `epochs=20` | Cuántas veces recorrer el corpus completo. |

#### Por qué `epochs=20` y no 30 (como en el notebook 11)

Aquí usamos un poco menos. Razones posibles:
- FastText es **más lento por época** (tiene que actualizar también los n-gramas).
- 20 es suficiente para converger.
- Es una decisión del autor del notebook — no hay una receta única.

#### Interpretando el output `(9125274, 10741780)`

El método `.train()` devuelve una **tupla con dos números**:

- **9.125.274**: número de palabras **efectivamente procesadas** (después del submuestreo).
- **10.741.780**: número total de palabras "raw" en el corpus a lo largo de las 20 épocas.

> 💡 **Submuestreo**: las palabras muy frecuentes (como artículos) se "saltan" probabilísticamente para no dominar el aprendizaje. Por eso hay menos efectivas que totales.

#### ¿Por qué tarda más que Word2Vec?

- Word2Vec actualiza **1 vector por palabra** (la palabra entera).
- FastText actualiza **muchos vectores por palabra** (todos sus n-gramas).

Por eso es más caro computacionalmente. El precio del superpoder de manejar OOV.


## Guardamos los modelos

### Guardar el modelo

Igual que en el notebook anterior. Entrenar tarda → guardamos el resultado para no tener que repetirlo.

#### Diferencia con Word2Vec

El archivo de FastText es **más grande** que el de Word2Vec porque guarda:
- Los embeddings de las 8770 palabras.
- Los embeddings de los 2 millones de n-gramas.

Para corpus grandes puede ser **gigabytes**. Para este pequeño corpus, son pocos MB.


In [13]:
ft_sg.save('./w2v_model_fast.pkl')


### El archivo `.pkl`

```python
ft_sg.save('./w2v_model_fast.pkl')
```

#### Sobre el nombre `w2v_model_fast.pkl`

⚠️ El nombre es **engañoso**: lleva `w2v` (Word2Vec) pero realmente es FastText. Es un copy-paste del notebook anterior. **En código limpio se llamaría** `ft_sg_model.pkl` o `fasttext_model.pkl`.

#### Cómo recargarlo más tarde

```python
from gensim.models import FastText
ft_sg = FastText.load('./w2v_model_fast.pkl')
```

Cargar es instantáneo.


## Algunos resultados

### A jugar con el modelo

Ahora vienen las consultas más interesantes. Vamos a comparar mentalmente los resultados con los de Word2Vec del notebook anterior:

| Consulta | Word2Vec (nb 11) | FastText (esperado) |
| --- | --- | --- |
| `most_similar('homer')` | `marge, depressed, abe, ...` (relaciones de contexto) | Palabras que **comparten letras** con homer |
| `most_similar('marge')` | `umm, badly, homer, ...` | Palabras parecidas en escritura |
| `similarity('maggie', 'baby')` | 0.63 (semánticamente cerca) | ? (puede ser menor) |
| `most_similar('asereje')` | ❌ KeyError | ✅ ¡Funcionará! |

🎯 **Lo crucial**: vamos a ver que **FastText es mejor para morfología/OOV**, pero **a veces peor para semántica pura**. Es un trade-off.


In [14]:
ft_sg.wv.most_similar(positive=["homer"])

[('knockahomer', 0.6203237771987915),
 ('homey', 0.6109655499458313),
 ('homeboy', 0.5451107621192932),
 ('hom', 0.5187292098999023),
 ('hometown', 0.5085053443908691),
 ('astronomer', 0.49521616101264954),
 ('homosexual', 0.4760384261608124),
 ('fonzie', 0.47000929713249207),
 ('simpson', 0.4674835801124573),
 ('home', 0.4665936529636383)]

### 🔍 Resultados para `homer` — COMPARA con el notebook 11

```
[('knockahomer', 0.62), ('homey', 0.61), ('homeboy', 0.55), ('hom', 0.52),
 ('hometown', 0.51), ('astronomer', 0.50), ('homosexual', 0.48),
 ('fonzie', 0.47), ('simpson', 0.47), ('home', 0.47)]
```

#### Lo que vemos

| Palabra | Similitud | Por qué |
| --- | --- | --- |
| `knockahomer` | 0.62 | Contiene `homer` literalmente. |
| `homey` | 0.61 | Comparte `home` con `homer`. |
| `homeboy` | 0.55 | Comparte `home`. |
| `hom` | 0.52 | El n-grama `hom` es parte de `homer`. |
| `hometown` | 0.51 | Comparte `home`. |
| `astronomer` | 0.50 | Comparte `omer`. |
| `homosexual` | 0.48 | Comparte `hom`. |
| `simpson` | 0.47 | Aquí sí, **semántica** (apellido de Homer). |

#### Comparación con Word2Vec (notebook 11)

Recuerda que en Word2Vec, `most_similar('homer')` dio:
```
marge, depressed, abe, y'ello, friendship, adopt, rude, eliza, babysitter, mr
```

**Word2Vec dio palabras semánticamente relacionadas** (marge=mujer de Homer, abe=padre).
**FastText da palabras ortográficamente relacionadas** (que comparten letras).

#### Esto NO es un fallo de FastText, es su naturaleza

FastText prioriza la **forma** de las palabras (n-gramas). Para tareas donde **la morfología importa** (typos, idiomas con flexión), esto es bueno. Para **semántica pura**, Word2Vec puede ser mejor.

> 💡 **Lección importante**: ningún modelo es perfecto para todo. Elige según el problema.

#### El caso de `simpson`

Que `simpson` aparezca con 0.47 nos dice que FastText **también captura algo de semántica** (Homer Simpson aparecen juntos en muchas frases), pero menos dominantemente que las relaciones morfológicas.


In [15]:
ft_sg.wv.most_similar(positive=["marge"])

[('sarge', 0.65062016248703),
 ('margarita', 0.5991832613945007),
 ('margie', 0.5977099537849426),
 ('margaret', 0.5758613348007202),
 ('barge', 0.5547234416007996),
 ('marmaduke', 0.517735481262207),
 ('marjorie', 0.48516175150871277),
 ('marco', 0.4706002473831177),
 ('mars', 0.45953020453453064),
 ('marble', 0.4571412205696106)]

### Resultados para `marge` — confirmación del patrón

```
[('sarge', 0.65), ('margarita', 0.60), ('margie', 0.60), ('margaret', 0.58),
 ('barge', 0.55), ('marmaduke', 0.52), ('marjorie', 0.49), ('marco', 0.47),
 ('mars', 0.46), ('marble', 0.46)]
```

#### Análisis

| Palabra | Por qué se parece a `marge` |
| --- | --- |
| `sarge` | Rima, comparten `arge`. |
| `margarita` | Comparte `marg`. |
| `margie` | Variante diminutiva, comparte `marg`. |
| `margaret` | Nombre completo, comparte `marg`. |
| `barge` | Rima, comparte `arge`. |
| `mar...` (mars, marco, marble) | Todas comparten `mar`. |

**FastText ha agrupado palabras que se ESCRIBEN parecido**, no que tengan que ver con el personaje Marge Simpson.

#### Hay menos información semántica que en Word2Vec

En Word2Vec, `marge` se parecía a `homer`, `flanders`, `honey` — **gente que aparece junto a ella**. Aquí FastText prioriza la morfología.

#### Sería ideal combinarlos

Algunos sistemas reales **combinan** los dos enfoques: aprenden tanto a nivel de palabra como de subpalabra. Pero eso es avanzado y va más allá de este notebook.


In [16]:
ft_sg.wv.most_similar(positive=["bart"])

[('barto', 0.5650189518928528),
 ('bartman', 0.5263718962669373),
 ('bartron', 0.49895814061164856),
 ('barty', 0.49754899740219116),
 ('baryshnikov', 0.4928662180900574),
 ('nikki', 0.4818361699581146),
 ('bartholomew', 0.4777153432369232),
 ('dart', 0.46039220690727234),
 ('impart', 0.45737800002098083),
 ('iq', 0.4564904272556305)]

### Resultados para `bart`

```
[('barto', 0.57), ('bartman', 0.53), ('bartron', 0.50), ('barty', 0.50),
 ('baryshnikov', 0.49), ('nikki', 0.48), ('bartholomew', 0.48),
 ('dart', 0.46), ('impart', 0.46), ('iq', 0.46)]
```

#### Observaciones

- `barto`, `bartman`, `bartron`, `barty`, `bartholomew` — todos contienen `bart`. ¡Lógico!
- `dart`, `impart` — comparten `art`.
- `baryshnikov` — empieza con `bar` (Mikhail Baryshnikov es bailarín ruso).
- `nikki` — sorpresa: probablemente captura algo del contexto (Bart conoce a una Nikki en algún episodio).

#### Notable ausencia: `lisa`

En Word2Vec, `bart` y `lisa` tenían similitud **0.74** (hermanos, casi siempre juntos). Aquí Lisa **no aparece en el top 10** porque las dos palabras son ortográficamente diferentes (`bart` vs `lisa` no comparten n-gramas).

> 🎯 **Trade-off claro**: FastText pierde algunas relaciones semánticas obvias a cambio de manejar mejor la morfología.


In [17]:
ft_sg.wv.similarity('maggie', 'baby')

np.float32(0.3153603)

### `similarity('maggie', 'baby') = 0.32`

#### Comparación crítica con Word2Vec

| Modelo | similarity(maggie, baby) |
| --- | --- |
| Word2Vec (notebook 11) | **0.63** ⬆️ |
| FastText (este notebook) | **0.32** ⬇️ |

#### Por qué cae a la mitad

`maggie` y `baby` son palabras **completamente distintas en escritura**:
- `maggie` → `<ma`, `mag`, `agg`, `ggi`, `gie`, `ie>`
- `baby` → `<ba`, `bab`, `aby`, `by>`

**No comparten n-gramas**. Por tanto, FastText las ve como muy distintas, aunque semánticamente lo sean.

#### ¿Es esto un problema?

Depende del uso:
- **Si tu objetivo es "ver qué palabras tratan del mismo tema"** → Word2Vec aquí es mejor.
- **Si tu objetivo es "ver qué palabras son variantes de la misma raíz"** → FastText brilla.

#### El número 0.32 sigue siendo positivo

No es despreciable. FastText aprendió **algo** de la relación semántica (porque ambas palabras aparecen en contextos similares — bebés, biberones, etc.). Pero la dominante en su criterio sigue siendo la forma de la palabra.


In [18]:
ft_sg.wv.similarity('bart', 'nelson')

np.float32(0.28543636)

### `similarity('bart', 'nelson') = 0.29`

#### Comparación con Word2Vec

| Modelo | similarity(bart, nelson) |
| --- | --- |
| Word2Vec | 0.45 |
| FastText | **0.29** ⬇️ |

#### Por qué baja

`bart` y `nelson` no comparten letras → poca similitud morfológica.

Pero **alguna semántica sí queda**: aparecen en frases similares (los dos son niños del cole en Springfield), por eso no llega a 0.

#### Patrón general que estamos viendo

Las similitudes **bajan** en FastText cuando las palabras son ortográficamente distintas, **independientemente** de su relación semántica.

#### ¿Cuándo importaría más una u otra?

| Aplicación | Modelo preferido |
| --- | --- |
| Buscar variantes de una palabra | FastText |
| Encontrar sinónimos verdaderos | Word2Vec |
| Manejar typos en input de usuarios | FastText |
| Análisis temático profundo | Word2Vec |
| Texto en idioma con mucha flexión (alemán, finés, latín) | FastText |
| Inglés moderno simple | Word2Vec o FastText, da igual |


In [19]:
ft_sg.wv.doesnt_match(['jimbo', 'milhouse', 'kearney'])

'milhouse'

### `doesnt_match(['jimbo', 'milhouse', 'kearney']) = 'milhouse'`

#### Quiénes son estos personajes

- **Jimbo Jones** y **Kearney Zzyzwicz**: los **bullies** del colegio (matones).
- **Milhouse**: el mejor amigo de Bart, **no es un bully**.

**El resultado es correcto**: Milhouse es el intruso porque es el único que no es matón.

#### ¿Cómo lo ha sabido FastText?

Recuerda cómo funciona `doesnt_match`:
1. Calcula el centroide (promedio) de los tres vectores.
2. Mide la distancia de cada uno al centroide.
3. Devuelve el más lejano.

Como Jimbo y Kearney aparecen juntos en contextos parecidos (escenas de bullying), sus vectores son similares. Milhouse aparece en otros contextos (con Bart, en su casa, etc.), su vector está más alejado.

> 💡 Aquí FastText **sí captura semántica** porque hay suficientes ejemplos en el corpus.


In [20]:
ft_sg.wv.doesnt_match(['homer', 'patty', 'selma'])

'homer'

### `doesnt_match(['homer', 'patty', 'selma']) = 'homer'`

#### El mismo resultado que en Word2Vec ✅

Igual que en el notebook anterior, el modelo identifica correctamente a Homer como el intruso. Patty y Selma son hermanas gemelas (siempre juntas), Homer es el outsider.

Esto demuestra que **ambos modelos (Word2Vec y FastText) capturan algo de las relaciones del corpus**, no solo morfología.


## Out-of-Vocabulary (OOV) Words

## 🎯 El gran momento: OOV en FastText

Aquí viene **la prueba de fuego** del notebook. Vamos a pedirle al modelo una palabra que **definitivamente no está en el corpus de Los Simpsons**: `asereje`.

Recuerda lo que pasó en Word2Vec:
```
KeyError: "Key 'asereje' not present in vocabulary"
```

¿Pasará lo mismo en FastText?


la cantidad de n-grams creados durante el entrenamiento del FastText hace improbable (que no imposible) que alguna palabra no pueda ser construída como una bolsa de n-grams

### Por qué FastText resuelve OOV — explicación detallada

#### El mecanismo

Cuando le pedimos a FastText el vector de `asereje`:

1. **Descompone la palabra en n-gramas** (entre min_n=3 y max_n=6):
   - 3-gramas: `<as`, `ase`, `ser`, `ere`, `rej`, `eje`, `je>`
   - 4-gramas: `<ase`, `aser`, `sere`, `erej`, `reje`, `eje>`
   - 5-gramas: `<aser`, `asere`, `serej`, `ereje`, `reje>`
   - 6-gramas: `<asere`, `asereje`, `sereje`, `ereje>`

2. **Para cada n-grama, busca su embedding aprendido durante el entrenamiento**.
   - Algunos n-gramas (`ser`, `ere`, `eje`) probablemente aparecieron en el corpus de Los Simpsons en palabras como `seriously`, `era`, etc.
   - Sus embeddings están aprendidos.

3. **Suma todos los embeddings de n-gramas** → obtiene el vector de `asereje`.

#### El resultado es un vector real

No es un placeholder. No es ruido. Es **un vector aprendido**, construido sumando piezas que sí conoce.

#### Calidad esperada

El vector de `asereje` será **una mezcla de los significados de sus n-gramas**. Por ejemplo:
- `ser` puede aparecer en `serious`, `service`, `series`...
- `eje` puede aparecer en `vegetable`, etc.

El vector resultante refleja **lo que tienen en común** las palabras donde aparecen esos n-gramas. **No es perfecto**, pero es **infinitamente mejor que un KeyError**.

#### Por qué la frase del notebook dice "improbable que no funcione"

Para que FastText **falle** completamente con una palabra, **TODOS sus n-gramas** tendrían que ser desconocidos. Esto es **muy improbable** porque:
- Los n-gramas son trozos cortos (3-6 letras).
- En un corpus razonable, casi cualquier 3-grama del idioma ha aparecido.
- Hay muchos n-gramas por palabra (≥10 normalmente).

Solo fallaría con palabras de un alfabeto completamente diferente (ej: caracteres chinos en un modelo entrenado solo con latinos).


In [21]:
'asereje' in ft_sg.wv.key_to_index

False

### Confirmación: `asereje` NO está en el vocabulario

```python
'asereje' in ft_sg.wv.key_to_index
# False
```

`key_to_index` es el diccionario interno de FastText con las **8770 palabras** del vocabulario. `asereje` no está → devuelve `False`.

> Lo importante viene ahora: aunque la palabra NO está en el vocabulario, FastText puede DEVOLVER un vector para ella usando los n-gramas. **¡La siguiente celda no dará error!**


In [22]:
ft_sg.wv.most_similar('asereje')

[('serenity', 0.6360535025596619),
 ('taser', 0.6092368364334106),
 ('eraser', 0.5883129239082336),
 ('laser', 0.5858527421951294),
 ('phaser', 0.5723046660423279),
 ('yayyy', 0.5605917572975159),
 ('unnecessary', 0.5492251515388489),
 ('cease', 0.5491967797279358),
 ('reject', 0.5441057682037354),
 ('sera', 0.5437067747116089)]

## ✨ EL MOMENTO MÁGICO

### `ft_sg.wv.most_similar('asereje')` ¡SÍ FUNCIONA!

```
[('serenity', 0.64), ('taser', 0.61), ('eraser', 0.59), ('laser', 0.59),
 ('phaser', 0.57), ('yayyy', 0.56), ('unnecessary', 0.55), ('cease', 0.55),
 ('reject', 0.54), ('sera', 0.54)]
```

### Compara con Word2Vec del notebook anterior

| Modelo | `most_similar('asereje')` |
| --- | --- |
| Word2Vec (notebook 11) | 💥 `KeyError: "Key 'asereje' not present in vocabulary"` |
| **FastText (este notebook)** | ✅ **Lista de palabras similares** |

### Análisis de los resultados

| Palabra | Similitud | Por qué |
| --- | --- | --- |
| `serenity` | 0.64 | Comparte `ser`, `ere`. |
| `taser` | 0.61 | Comparte `aser`. |
| `eraser` | 0.59 | Comparte `ase`, `ser`. |
| `laser` | 0.59 | Comparte `ase`, `ser`. |
| `phaser` | 0.57 | Comparte `aser`, `ser`. |
| `unnecessary` | 0.55 | Comparte `eces`, `ces`... |
| `cease` | 0.55 | Comparte `eas`, `ase`. |
| `reject` | 0.54 | Comparte `eje`. |

**FastText ha encontrado palabras que comparten n-gramas** con `asereje`. Aunque la palabra nunca apareció, el modelo "reconstruye" su vector y encuentra vecinos plausibles.

### ¿Es esto útil?

**Depende del contexto**:

- **Si necesitas un vector "razonable" para una palabra desconocida** → ✅ FastText resuelve el problema sin fallar.
- **Si necesitas que el vector capture el significado real** (un baile español, una canción) → ❌ FastText no puede inventar significado a partir de letras.

### Cuándo esto SÍ es valioso

- **Typos**: `homerr` → el modelo devuelve algo muy parecido a `homer`. ✅
- **Variantes morfológicas**: `runs` → similar a `run`. ✅
- **Palabras nuevas en el mismo idioma**: `pokemon` (jamás visto) → algún vecino plausible. ✅

### Cuándo NO es tan valioso

- **Palabras extranjeras**: `asereje` (español) en modelo entrenado en inglés → vecinos por casualidad léxica.
- **Slang completamente nuevo**: si no comparte n-gramas con nada conocido, los vecinos serán azarosos.


In [23]:
ft_sg.wv['asereje'].shape

(300,)

### Verificación final: el vector tiene la forma correcta

```python
ft_sg.wv['asereje'].shape
# (300,)
```

#### Qué significa `(300,)`

Es la **forma** (shape) del vector: un array unidimensional con **300 elementos**.

Esto es EXACTAMENTE el mismo shape que para palabras que sí estaban en el vocabulario (como `homer`, `marge`). FastText no distingue: te da un vector de 300 números, vengan de un embedding aprendido directamente o construido sumando n-gramas.

#### Lo que esto significa en la práctica

Puedes usar `asereje` **igual que cualquier otra palabra** en cualquier pipeline downstream:
- Calcular similitudes.
- Promediar vectores para representar frases.
- Pasar a un clasificador.
- Buscar vecinos cercanos.

**No hay caso especial que manejar**. El código que usabas con Word2Vec necesitaba `try/except` para palabras OOV; con FastText, no.

---

# 🏁 Resumen final del notebook

## Lo que hemos aprendido

### Conceptos

| Concepto | Idea de una línea |
| --- | --- |
| **FastText** | Word2Vec mejorado: aprende vectores de n-gramas de caracteres en lugar de palabras enteras. |
| **n-grama de caracteres** | Trozo consecutivo de N letras dentro de una palabra. |
| **`min_n` y `max_n`** | Longitudes mínima y máxima de n-gramas que considera. |
| **OOV resuelto** | Para palabras nuevas, FastText suma los vectores de sus n-gramas. |
| **Trade-off** | FastText prioriza morfología sobre semántica pura. |

### Comparativa Word2Vec vs FastText (este notebook vs el anterior)

| Característica | Word2Vec | FastText |
| --- | --- | --- |
| Trabaja con | Palabras enteras | n-gramas de caracteres |
| Vocabulario explícito | 3239 (con min_count=20) | 8770 (con min_count=5) |
| OOV | ❌ KeyError | ✅ Construye vector |
| `similarity(maggie, baby)` | 0.63 | 0.32 |
| `most_similar('homer')` | marge, depressed, abe... | knockahomer, homey, homeboy... |
| Velocidad de entrenamiento | Más rápido | Más lento |
| Memoria | Menos | Más |
| Idiomas con flexión | Mediocre | Excelente |
| Manejo de typos | Pobre | Excelente |

## Cuándo elegir FastText vs Word2Vec

| Si tu problema tiene... | Elige |
| --- | --- |
| Texto limpio, en inglés, semántica importante | **Word2Vec** |
| Typos, slang, abreviaturas | **FastText** |
| Idioma con mucha flexión (español, alemán, finés) | **FastText** |
| Palabras nuevas frecuentes (nombres propios, jerga técnica) | **FastText** |
| Recursos limitados (memoria, tiempo) | **Word2Vec** |
| Quieres maximizar generalización | **FastText** |

## Mejoras y siguientes pasos

| Mejora | Esfuerzo | Beneficio |
| --- | --- | --- |
| Probar `min_n=2, max_n=7` (más rango de n-gramas) | Trivial | Mejor cobertura morfológica. |
| Probar embeddings preentrenados de FastText (Facebook publica modelos en 157 idiomas) | Bajo | Calidad espectacular sin entrenar. |
| Usar **subword embeddings con BPE** (Byte Pair Encoding) | Medio | El estándar moderno (GPT, BERT lo usan). |
| Saltar a transformers (BERT, RoBERTa) | Alto | Estado del arte para casi todo. |

## Aplicaciones reales de FastText

- **Buscadores tolerantes a errores**: encuentras "hommer" aunque escribas mal "homer".
- **Clasificación de textos** con vocabulario diverso.
- **Idiomas con poca data**: FastText comparte información entre palabras relacionadas morfológicamente.
- **Análisis de redes sociales** (mucho slang, typos, neologismos).
- **Sistemas multilingües**: Facebook usa FastText para representar 157 idiomas con un modelo unificado.

## Reflexión final

> 🎯 **Mensaje para gente que acaba de empezar a programar**:
>
> Has visto en dos notebooks (11 y 12) la evolución de las ideas en NLP. Word2Vec fue revolucionario en 2013. FastText lo mejoró en 2016. Hoy hay modelos aún mejores (BERT, GPT) que llevan estas ideas mucho más lejos.
>
> **Pero entender estos dos modelos es entender la base de TODO** lo que vino después. Los conceptos que has aprendido (embeddings, contexto, n-gramas, OOV) están en TODOS los modelos modernos.
>
> No te preocupes si no entendiste cada detalle matemático. Lo importante es que **sepas qué hacen los modelos, por qué existen, y cuándo elegir cada uno**.

¡Felicidades por completar la sección de embeddings!
